In [ ]:
# M11 — Temporal Validation

##Cell 1: Setup & Imports

In [ ]:
# ═══════════════════════════════════════════════════════════
# M11 — Temporal Validation · Cell 1: Setup & Imports
# ═══════════════════════════════════════════════════════════
import importlib, sys, os, json, warnings
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings('ignore')

# from google.colab import drive
# drive.mount('/content/drive')
src_dir = '/content/drive/MyDrive/0000_Kaan_CDS/src'
os.chdir(src_dir)
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

MODULE_NAMES = ['plot_style', 'cds_config', 'm1_calibration', 'm2_thresholds',
                'm3_errors', 'm4_e2e', 'm5_shap', 'm6_conformal', 'm7_cascade',
                'm8_cds_report']

for mod_name in MODULE_NAMES:
    spec = importlib.util.spec_from_file_location(
        mod_name, os.path.join(src_dir, f"{mod_name}.py"))
    mod = importlib.util.module_from_spec(spec)
    sys.modules[mod_name] = mod
    spec.loader.exec_module(mod)

from cds_config import *
setup_notebook("Chat 11 — M11 Temporal Validation")
from plot_style import PALETTE, save_fig, despine_all, add_panel_label, CHOSEN_FONT
from m1_calibration import load_calibrated_probs
from m2_thresholds import load_threshold_config, get_operating_point
from m6_conformal import fit_conformal, predict_sets, compute_aps_scores, compute_coverage

import joblib

# ── Paths ──
CDS  = Path('/content/drive/MyDrive/0000_Kaan_CDS')
PUB  = CDS / 'publication_tables_figures'
TAB_DIR = PUB / 'tables'
FIG_DIR = PUB / 'figures'
MODULES = CDS / 'modules'

# M11-specific output
M11_DIR = CDS / 'modules' / 'm11_temporal'
M11_DATA = M11_DIR / 'data'
M11_ANALYSIS = M11_DIR / 'analysis'
M11_PLOTS = M11_DIR / 'plots'

for d in [M11_DIR, M11_DATA, M11_ANALYSIS, M11_PLOTS]:
    d.mkdir(parents=True, exist_ok=True)

# ── Frozen assets ──
FROZEN_MODELS = CDS / 'frozen_models'
TH_CONFIG_PATH = MODULES / 'm2_thresholds/configs/threshold_config.json'
CAL_PATH = MODULES / 'm1_calibration/calibrators/stage1_cbc_only_calibrator.joblib'
CONF_SUMMARY_PATH = MODULES / 'm6_conformal/metrics/conformal_summary.xlsx'

# Load frozen configs
with open(TH_CONFIG_PATH) as f:
    th_config = json.load(f)

calibrator_dict = joblib.load(CAL_PATH)
calibrator_s1_cbc = calibrator_dict['calibrator']  # IsotonicRegression

conf_summary = pd.read_excel(CONF_SUMMARY_PATH)
# Extract frozen q_hat values (randomized, α=0.10 — primary)
Q_HATS = {}
for _, row in conf_summary[conf_summary['method'] == 'randomized'].iterrows():
    key = f"{row['scenario'].strip()}_S{row['stage']}_a{row['alpha']}"
    Q_HATS[key] = row['q_hat']

print("✅ All modules imported")
print(f"✅ M11 output → {M11_DIR}")
print(f"✅ Calibrator: {type(calibrator_s1_cbc).__name__}")
print(f"✅ Frozen q_hats: {json.dumps({k: round(v,6) for k,v in Q_HATS.items()}, indent=2)}")
print(f"✅ Threshold config loaded (v={th_config['version']})")



# ── Load original test probabilities (same as M9 Cell 3) ──
prob = {}
for sc in ['CBC_Only', 'CBC_BIO']:
    for st in ['1', '2']:
        for sp in ['oof', 'test']:
            prob[f"S{st}_{sc}_{sp}"] = load_calibrated_probs(PATHS, st, sc, sp)

print(f"✅ Loaded {len(prob)} probability sets")

##Cell 2 — Load OOF & Build Diagnosis Mapping

In [ ]:
# ═══════════════════════════════════════════════════════════
# M11 · Cell 2: Load OOF data & build label mappings
# ═══════════════════════════════════════════════════════════

# Load all OOF parquets
s1_cbc_oof = load_calibrated_probs(PATHS, '1', 'CBC_Only', 'oof')
s1_bio_oof = load_calibrated_probs(PATHS, '1', 'CBC_BIO', 'oof')
s2_cbc_oof = load_calibrated_probs(PATHS, '2', 'CBC_Only', 'oof')
s2_bio_oof = load_calibrated_probs(PATHS, '2', 'CBC_BIO', 'oof')

print(f"S1 CBC OOF: {s1_cbc_oof.shape}  (OAC={sum(s1_cbc_oof['target']==0)}, AAC={sum(s1_cbc_oof['target']==1)})")
print(f"S2 CBC OOF: {s2_cbc_oof.shape}  classes={s2_cbc_oof['target'].value_counts().sort_index().to_dict()}")

# ── Diagnosis → 4-class mapping (from S2 OOF which has both) ──
diag_to_label = s2_cbc_oof.groupby('diagnosis')['target_label'].first().to_dict()
diag_to_target = s2_cbc_oof.groupby('diagnosis')['target'].first().to_dict()
print(f"\nDiagnosis → 4-class mapping (from S2 AAC patients):")
for d, l in sorted(diag_to_label.items()):
    print(f"  {d:20s} → {l} (target={diag_to_target[d]})")

# ── Check OAC patients' diagnoses ──
das_patients = s1_bio_oof[s1_bio_oof['target'] == 0].copy()
print(f"\nOAC patients (n={len(das_patients)}) diagnosis distribution:")
print(das_patients['diagnosis'].value_counts().to_string())

# ── Check: can we map ALL OAC diagnoses to 4-class? ──
unmapped = set(das_patients['diagnosis'].unique()) - set(diag_to_label.keys())
if unmapped:
    print(f"\n⚠️ Unmapped OAC diagnoses: {unmapped}")
    print("   → Need manual mapping for these")
else:
    print(f"\n✅ All OAC diagnoses have 4-class mapping")

# ── Identify BASE features (raw lab values, not ratios) ──
all_s1_bio_cols = list(s1_bio_oof.columns)
ratio_cols = [c for c in all_s1_bio_cols if '_div_' in c]
meta_cols = ['target', 'diagnosis', 'prob_DAS', 'prob_IAS', 'pred', 'correct',
             'confidence', 'prob_IAS_cal', 'prob_DAS_cal', 'calibration_method']
base_features = [c for c in all_s1_bio_cols if c not in meta_cols and c not in ratio_cols]

print(f"\nBase features ({len(base_features)}):")
print(f"  {base_features}")
print(f"Ratio features ({len(ratio_cols)}):")
print(f"  {ratio_cols[:5]} ... (total {len(ratio_cols)})")

##Cell 3 — Generate Synthetic Temporal Cohort

In [ ]:
# ═══════════════════════════════════════════════════════════
# M11 · Cell 3: Load Temporal Data (3-tier: Real → Synthetic → Generate)
# ═══════════════════════════════════════════════════════════
np.random.seed(42)


if 'FEAT_NAMES' not in dir() or not FEAT_NAMES:
    FEAT_NAMES = {}
    for d in sorted(FROZEN_MODELS.iterdir()):
        if d.is_dir():
            FEAT_NAMES[d.name] = joblib.load(d / 'feature_names.joblib')

    EXP_MAP = {
        '20260201_1719_Stage1_CBC_Only': ('S1', 'CBC_Only'),
        '20260201_2035_Stage2_CBC_Only': ('S2', 'CBC_Only'),
        '20260202_0436_Stage1_CBC_BIO':  ('S1', 'CBC_BIO'),
        '20260202_0507_Stage2_CBC_BIO':  ('S2', 'CBC_BIO'),
    }
    print(f"✅ Loaded feature names for {len(FEAT_NAMES)} frozen models")




REAL_PATH  = M11_DATA / 'temporal_cohort_real.xlsx'
SYNTH_PATH = M11_DATA / 'temporal_cohort_template_FILLED.xlsx'

if REAL_PATH.exists():
    DATA_MODE = 'REAL'
    temporal = pd.read_excel(REAL_PATH, sheet_name='Temporal_Data', header=0, skiprows=[1, 2])
    print(f"🟢 REAL temporal data → {len(temporal)} patients")

elif SYNTH_PATH.exists():
    DATA_MODE = 'SYNTHETIC_EXCEL'
    temporal = pd.read_excel(SYNTH_PATH, sheet_name='Temporal_Data', header=0, skiprows=[1, 2])
    print(f"🟡 Synthetic Excel → {len(temporal)} patients")
    print(f"   ⚠️ SYNTHETIC — never publish")

else:
    DATA_MODE = 'SYNTHETIC_OOF'
    print(f"🟠 No Excel found — generating from OOF bootstrap...")

    # ── Source: S1 BIO OOF (AAC only) + S2 labels ──
    source_s1 = s1_bio_oof[s1_bio_oof['target'] == 1].copy().reset_index(drop=True)
    source_s2 = s2_bio_oof[['target', 'target_label', 'diagnosis']].copy().reset_index(drop=True)
    source = source_s1.copy()
    source['target'] = source_s2['target'].values
    source['target_label'] = source_s2['target_label'].values
    source['diagnosis'] = source_s2['diagnosis'].values

    # Bootstrap ~150
    TARGET_N = 150
    class_ns = {0: 49, 1: 30, 2: 40, 3: 31}
    sampled = []
    for cls, n in class_ns.items():
        pool = source[source['target'] == cls]
        idx = np.random.choice(pool.index, size=n, replace=True)
        sampled.append(source.loc[idx])
    temporal = pd.concat(sampled, ignore_index=True)

    # Noise
    meta_cols = ['target', 'target_label', 'diagnosis',
                 'prob_DAS', 'prob_IAS', 'pred', 'correct', 'confidence',
                 'prob_IAS_cal', 'prob_DAS_cal', 'calibration_method']
    all_cols = list(temporal.columns)
    ratio_cols = [c for c in all_cols if '_div_' in c]
    base_features = [c for c in all_cols if c not in meta_cols and c not in ratio_cols]

    BOUNDS = {
        'mcv_f_l': (50, 130), 'ret_he_pg': (10, 45), 'rbc_10_6_u_l': (2.0, 8.0),
        'ret_number_10_6_l': (0.001, 1.0), 'rdw_sd_fl': (25, 90),
        'delta_he_pg': (-10, 15), 'frc_perc': (0, 100), 'mchc_g_dl': (25, 40),
        'irf_pct': (0, 50), 'nrbc_pct': (0, 20), 'hgb_g_d_l': (4, 20),
        'micro_macro_ratio': (0, 100), 'yas': (18, 95),
        'ferritin': (1, 1500), 'demir': (5, 400), 'ldh': (50, 800), 'uibc': (50, 600),
    }
    for feat in base_features:
        if feat == 'yas':
            temporal[feat] = temporal[feat] + np.random.randint(-2, 3, size=len(temporal))
        else:
            temporal[feat] = temporal[feat] * (1 + np.random.normal(0, 0.07, size=len(temporal)))
        if feat in BOUNDS:
            temporal[feat] = temporal[feat].clip(*BOUNDS[feat])
        if feat in ['demir', 'ldh', 'uibc', 'yas']:
            temporal[feat] = temporal[feat].round().astype(int)

    # Drop old predictions
    drop = ['prob_DAS', 'prob_IAS', 'pred', 'correct', 'confidence',
            'prob_IAS_cal', 'prob_DAS_cal', 'calibration_method']
    temporal = temporal.drop(columns=[c for c in drop if c in temporal.columns])

    print(f"   Generated {len(temporal)} patients from OOF bootstrap")
    print(f"   ⚠️ SYNTHETIC — never publish")


# ── Force numeric dtypes on all features ──
NUMERIC_FEATS = [
    'mcv_f_l', 'ret_he_pg', 'rbc_10_6_u_l', 'ret_number_10_6_l',
    'rdw_sd_fl', 'delta_he_pg', 'frc_perc', 'mchc_g_dl',
    'irf_pct', 'nrbc_pct', 'hgb_g_d_l', 'micro_macro_ratio',
    'ferritin', 'demir', 'ldh', 'uibc', 'yas'
]

print(f"\nDtype check BEFORE coercion:")
for c in NUMERIC_FEATS:
    if c in temporal.columns:
        print(f"  {c}: {temporal[c].dtype}")



# ═══════════════════════════════════════════════════════════
# UNIT SANITY CHECK — apply training-scale conversions
# Training used R-code transformations:
#   ret_number_10_6_l = ret_number_10_9_l / 1000
#   frc_perc          = frc_number_10_6_u_l / rbc_10_6_u_l  (equivalent to FRC% / 100)
#   micro_macro_ratio = micro_r_percent / macro_r_percent
# Excel inputs may arrive in Sysmex raw units — detect and convert.
# ═══════════════════════════════════════════════════════════

# Reference ranges from training OOF
TRAIN_RANGES = {
    'frc_perc':          (0.0, 0.25),    # unitless ratio (FRC#/RBC)
    'ret_number_10_6_l': (0.0, 0.80),    # 10^6/L
    'micro_macro_ratio': (0.0, 410.0),   # unitless ratio
}

# Detect and auto-correct
for feat, (lo, hi) in TRAIN_RANGES.items():
    if feat not in temporal.columns:
        continue
    current_max = temporal[feat].max()
    current_mean = temporal[feat].mean()

    if feat == 'frc_perc' and current_max > 1.0:
        # Sysmex FRC% (0–10 scale) → ÷100
        print(f"⚠️  frc_perc max={current_max:.3f} looks like Sysmex FRC% — dividing by 100")
        temporal[feat] = temporal[feat] / 100

    elif feat == 'ret_number_10_6_l' and current_max > 5.0:
        # Sysmex RET# (10^9/L, values ~1–900) → ÷1000
        print(f"⚠️  ret_number_10_6_l max={current_max:.3f} looks like 10^9/L — dividing by 1000")
        temporal[feat] = temporal[feat] / 1000

    elif feat == 'micro_macro_ratio' and current_max > 1000:
        print(f"⚠️  micro_macro_ratio max={current_max:.3f} looks wrong — check source")

    new_max = temporal[feat].max()
    in_range = new_max <= hi * 1.5
    print(f"  {feat}: max={new_max:.4f} (training max={hi:.4f}) {'✅' if in_range else '⚠️'}")

# Force all to numeric, invalid → NaN
for c in NUMERIC_FEATS:
    if c in temporal.columns:
        temporal[c] = pd.to_numeric(temporal[c], errors='coerce')

# Target also
temporal['target'] = pd.to_numeric(temporal['target'], errors='coerce').astype('Int64')

# Check nulls after coercion
null_summary = temporal[NUMERIC_FEATS].isnull().sum()
null_cols = null_summary[null_summary > 0]
if len(null_cols) > 0:
    print(f"\n⚠️ Null values after numeric coercion:")
    print(null_cols.to_string())

# Int types for specific cols (after NaN handling)
for col in ['demir', 'ldh', 'uibc', 'yas']:
    if col in temporal.columns and temporal[col].notna().all():
        temporal[col] = temporal[col].round().astype(int)

# Drop rows with ANY null target
before = len(temporal)
temporal = temporal[temporal['target'].notna()].reset_index(drop=True)
temporal['target'] = temporal['target'].astype(int)
if before != len(temporal):
    print(f"\n⚠️ Dropped {before - len(temporal)} rows with missing target")

print(f"\n✅ Clean numeric dtypes, n={len(temporal)}")
# ── Compute ratio features ──
all_ratio_names = set()
for exp_name in FEAT_NAMES:
    for f in FEAT_NAMES[exp_name]:
        if '_div_' in f:
            all_ratio_names.add(f)

for rc in sorted(all_ratio_names):
    parts = rc.split('_div_')
    if len(parts) == 2:
        num, den = parts
        if num in temporal.columns and den in temporal.columns:
            temporal[rc] = temporal[num] / temporal[den].replace(0, np.nan)

# ── Identifiers ──
temporal['patient_idx'] = range(len(temporal))
temporal['cohort'] = DATA_MODE.lower()

# ── Verify frozen model features ──
print(f"\n✅ {DATA_MODE} cohort: {temporal.shape}")
print(f"   Class dist: {temporal['target'].value_counts().sort_index().to_dict()}")
for exp_name, (stage, scenario) in EXP_MAP.items():
    feats = FEAT_NAMES[exp_name]
    miss = [f for f in feats if f not in temporal.columns]
    print(f"   {stage} {scenario}: {'✅' if not miss else f'⚠️ {miss}'}")

temporal.to_parquet(M11_DATA / 'temporal_cohort.parquet', index=False)
print(f"\n✅ Saved → temporal_cohort.parquet")

## Cell 4: Frozen Model Predictions on Temporal Cohort

In [ ]:
!pip install -q autogluon.tabular

In [ ]:
# ═══════════════════════════════════════════════════════════
# M11 · Cell 4: Frozen Model Predictions on Temporal Cohort
# ═══════════════════════════════════════════════════════════
from autogluon.tabular import TabularPredictor

temporal = pd.read_parquet(M11_DATA / 'temporal_cohort.parquet')

# ── Load frozen models & feature names ──
MODEL_DIRS = {}
FEAT_NAMES = {}
for d in sorted(FROZEN_MODELS.iterdir()):
    if d.is_dir():
        name = d.name
        MODEL_DIRS[name] = d
        FEAT_NAMES[name] = joblib.load(d / 'feature_names.joblib')

EXP_MAP = {
    '20260201_1719_Stage1_CBC_Only': ('S1', 'CBC_Only'),
    '20260201_2035_Stage2_CBC_Only': ('S2', 'CBC_Only'),
    '20260202_0436_Stage1_CBC_BIO':  ('S1', 'CBC_BIO'),
    '20260202_0507_Stage2_CBC_BIO':  ('S2', 'CBC_BIO'),
}

# ── Predict with all 4 frozen models ──
predictions = {}

for exp_name, (stage, scenario) in EXP_MAP.items():
    model_dir = MODEL_DIRS[exp_name] / 'AutoGluon_Model'
    feat_names = FEAT_NAMES[exp_name]

    predictor = TabularPredictor.load(str(model_dir))
    X = temporal[feat_names].copy()
    probs = predictor.predict_proba(X)
    preds = predictor.predict(X)

    predictions[f"{stage}_{scenario}"] = {
        'probs': probs,
        'preds': preds,
        'features': feat_names,
    }
    print(f"  ✅ {stage} {scenario}: {len(probs)} patients, "
          f"pred classes={sorted(preds.unique())}")

# ── Build S1 result DataFrames (binary: OAC=0, AAC=1) ──
CLASS_NAMES_S2 = ['DEA', 'HA', 'HGB_HTZ', 'Normal']
CLASS_LABELS = {0: 'DEA', 1: 'HA', 2: 'HGB_HTZ', 3: 'Normal'}

for scenario in ['CBC_Only', 'CBC_BIO']:
    key = f"S1_{scenario}"
    probs = predictions[key]['probs']

    df = temporal[['patient_idx', 'target', 'target_label', 'diagnosis']].copy()
    # S1 binary target: all temporal patients are AAC (sampled from S2 OOF)
    df['s1_target'] = 1  # all AAC
    df['prob_DAS'] = probs[0].values
    df['prob_IAS'] = probs[1].values
    df['pred'] = predictions[key]['preds'].values

    # Calibration
    if scenario == 'CBC_Only':
        df['prob_IAS_cal'] = calibrator_s1_cbc.predict(df['prob_IAS'].values)
        df['prob_DAS_cal'] = 1 - df['prob_IAS_cal']
        df['calibration_method'] = 'isotonic'
    else:
        df['prob_IAS_cal'] = df['prob_IAS']
        df['prob_DAS_cal'] = df['prob_DAS']
        df['calibration_method'] = 'uncalibrated'

    df['confidence'] = df[['prob_DAS_cal', 'prob_IAS_cal']].max(axis=1)
    df['correct'] = (df['pred'] == df['s1_target']).astype(int)

    # Add features
    for feat in predictions[key]['features']:
        df[feat] = temporal[feat].values

    predictions[key]['df'] = df

    s1_acc = df['correct'].mean()
    n_ias = (df['pred'] == 1).sum()
    n_das = (df['pred'] == 0).sum()
    print(f"\n  {key}: S1 accuracy = {s1_acc:.3f}")
    print(f"    pred → AAC={n_ias}, OAC={n_das} (true: all AAC)")

# ── Build S2 result DataFrames (4-class) ──
for scenario in ['CBC_Only', 'CBC_BIO']:
    key = f"S2_{scenario}"
    probs = predictions[key]['probs']

    df = temporal[['patient_idx', 'target', 'target_label', 'diagnosis']].copy()
    for ci, cname in enumerate(CLASS_NAMES_S2):
        df[f'prob_{cname}'] = probs[ci].values
        df[f'prob_{cname}_cal'] = probs[ci].values  # S2 uncalibrated

    df['pred'] = predictions[key]['preds'].values
    df['pred_label'] = df['pred'].map(CLASS_LABELS)
    df['correct'] = (df['pred'] == df['target']).astype(int)
    df['confidence'] = df[[f'prob_{c}_cal' for c in CLASS_NAMES_S2]].max(axis=1)
    df['calibration_method'] = 'uncalibrated'

    # Add features
    for feat in predictions[key]['features']:
        df[feat] = temporal[feat].values

    predictions[key]['df'] = df

    acc = df['correct'].mean()
    per_class = df.groupby('target')['correct'].mean()
    print(f"\n  {key}: S2 accuracy = {acc:.3f}")
    print(f"    per-class: {per_class.to_dict()}")

# ── Save ──
for key in ['S1_CBC_Only', 'S1_CBC_BIO', 'S2_CBC_Only', 'S2_CBC_BIO']:
    path = M11_DATA / f'temporal_{key}_predictions.parquet'
    predictions[key]['df'].to_parquet(path, index=False)

print(f"\n✅ All 4 models predicted & saved")

## Cell 5 — Apply Thresholds, Zones & Cascade Simulation

In [ ]:
# ═══════════════════════════════════════════════════════════
# M11 · Cell 5: Cascade Simulation on Temporal Cohort
# ═══════════════════════════════════════════════════════════

# ── Frozen thresholds ──
s1_th_cbc = th_config['scenarios']['CBC_Only']['stage1']['threshold']
s1_th_bio = th_config['scenarios']['CBC_BIO']['stage1']['threshold']
s2_zones_cbc = th_config['scenarios']['CBC_Only']['stage2']
s2_zones_bio = th_config['scenarios']['CBC_BIO']['stage2']

print(f"Frozen thresholds:")
print(f"  S1 CBC_Only: {s1_th_cbc},  S1 CBC_BIO: {s1_th_bio}")
print(f"  S2 CBC_Only zones: LOW<{s2_zones_cbc['zone_low']}, HIGH≥{s2_zones_cbc['zone_high']}")
print(f"  S2 CBC_BIO  zones: LOW<{s2_zones_bio['zone_low']}, HIGH≥{s2_zones_bio['zone_high']}")

# ── Build cascade dataframe ──
s1_df = predictions['S1_CBC_Only']['df'].copy()
s2_cbc_df = predictions['S2_CBC_Only']['df'].copy()
s2_bio_df = predictions['S2_CBC_BIO']['df'].copy()

n_temporal = len(temporal)
cascade = temporal[['patient_idx', 'target', 'target_label', 'diagnosis']].copy()
cascade['true_label'] = cascade['target']
_disp_label = {'DEA': 'IDA', 'HGB_HTZ': 'HGB HTZ'}  # display only
cascade['true_label_name'] = cascade['target_label'].replace(_disp_label)

# ── TIER 1: S1 CBC_Only → S2 CBC_Only ──
cascade['s1_prob_IAS_cal'] = s1_df['prob_IAS_cal'].values
cascade['s1_is_IAS'] = (cascade['s1_prob_IAS_cal'] >= s1_th_cbc).astype(int)

# S2 CBC_Only predictions & zones
cascade['tier1_pred'] = s2_cbc_df['pred'].values
cascade['tier1_pred_label'] = s2_cbc_df['pred_label'].values
cascade['tier1_confidence'] = s2_cbc_df['confidence'].values

def assign_zone(conf, zone_low, zone_high):
    if conf >= zone_high:
        return 'HIGH'
    elif conf >= zone_low:
        return 'MEDIUM'
    else:
        return 'LOW'

cascade['tier1_zone'] = cascade['tier1_confidence'].apply(
    lambda c: assign_zone(c, s2_zones_cbc['zone_low'], s2_zones_cbc['zone_high']))

# ── Escalation logic ──
cascade['escalated'] = False
cascade['escalation_reason'] = 'none'

# OAC patients (S1 says not AAC) → escalate
das_mask = cascade['s1_is_IAS'] == 0
cascade.loc[das_mask, 'escalated'] = True
cascade.loc[das_mask, 'escalation_reason'] = 'S1_OAC_excluded'
cascade.loc[das_mask, 'tier1_zone'] = 'Excluded'

# MEDIUM/LOW zone (S2 uncertain) → escalate
med_low_mask = (~das_mask) & (cascade['tier1_zone'].isin(['MEDIUM', 'LOW']))
cascade.loc[med_low_mask, 'escalated'] = True
cascade.loc[med_low_mask, 'escalation_reason'] = 'S2_low_confidence'

# ── TIER 2: S2 CBC_BIO for escalated patients ──
cascade['tier2_pred'] = np.nan
cascade['tier2_pred_label'] = ''
cascade['tier2_confidence'] = np.nan
cascade['tier2_zone'] = ''

esc_mask = cascade['escalated']
cascade.loc[esc_mask, 'tier2_pred'] = s2_bio_df.loc[esc_mask, 'pred'].values
cascade.loc[esc_mask, 'tier2_pred_label'] = s2_bio_df.loc[esc_mask, 'pred_label'].values
cascade.loc[esc_mask, 'tier2_confidence'] = s2_bio_df.loc[esc_mask, 'confidence'].values
cascade.loc[esc_mask, 'tier2_zone'] = cascade.loc[esc_mask, 'tier2_confidence'].apply(
    lambda c: assign_zone(c, s2_zones_bio['zone_low'], s2_zones_bio['zone_high']))

# ── Final decision ──
cascade['final_pred'] = cascade['tier1_pred']
cascade['final_pred_label'] = cascade['tier1_pred_label']
cascade['final_zone'] = cascade['tier1_zone']
cascade['final_tier'] = 1

cascade.loc[esc_mask, 'final_pred'] = cascade.loc[esc_mask, 'tier2_pred']
cascade.loc[esc_mask, 'final_pred_label'] = cascade.loc[esc_mask, 'tier2_pred_label']
cascade.loc[esc_mask, 'final_zone'] = cascade.loc[esc_mask, 'tier2_zone']
cascade.loc[esc_mask, 'final_tier'] = 2

cascade['final_pred'] = cascade['final_pred'].astype(int)
cascade['final_tier'] = cascade['final_tier'].astype(int)
cascade['correct'] = (cascade['final_pred'] == cascade['true_label']).astype(int)

# ── Summary ──
n_esc = esc_mask.sum()
n_tier1 = n_temporal - n_esc
tier1_acc = cascade[cascade['final_tier'] == 1]['correct'].mean() if n_tier1 > 0 else 0
tier2_acc = cascade[cascade['final_tier'] == 2]['correct'].mean() if n_esc > 0 else 0
cascade_acc = cascade['correct'].mean()

print(f"\n{'='*60}")
print(f"TEMPORAL CASCADE SIMULATION RESULTS")
print(f"{'='*60}")
print(f"  Total patients:     {n_temporal}")
print(f"  Tier 1 retained:    {n_tier1} ({n_tier1/n_temporal*100:.1f}%)")
print(f"  Tier 1 accuracy:    {tier1_acc:.3f}")
print(f"  Escalated → Tier 2: {n_esc} ({n_esc/n_temporal*100:.1f}%)")
print(f"  Tier 2 accuracy:    {tier2_acc:.3f}")
print(f"  CASCADE accuracy:   {cascade_acc:.3f}")

print(f"\n  Escalation reasons:")
print(cascade['escalation_reason'].value_counts().to_string())

print(f"\n  Final zone distribution:")
print(cascade['final_zone'].value_counts().to_string())

print(f"\n  Per-class cascade accuracy:")
per_cls = cascade.groupby('true_label_name')['correct'].agg(['mean', 'count'])
print(per_cls.to_string())

# Save
cascade.to_parquet(M11_DATA / 'temporal_cascade_simulation.parquet', index=False)
print(f"\n✅ Cascade saved")

In [ ]:
# ═══════════════════════════════════════════════════════════
# M11 · Cell 5b: Conformal Prediction (fix tuple unpacking)
# ═══════════════════════════════════════════════════════════

for scenario, key_prefix in [('CBC_Only', 'tier1'), ('CBC_BIO', 'tier2')]:
    q_key = f"{scenario}_S2_a0.1"
    q_hat = Q_HATS[q_key]

    s2_df = predictions[f'S2_{scenario}']['df']
    probs_cal = s2_df[[f'prob_{c}_cal' for c in CLASS_NAMES_S2]].values

    result = predict_sets(probs_cal, q_hat, randomized=True,
                          rng=np.random.default_rng(42))
    # Unpack tuple: (list_of_sets, sizes_array)
    sets_list = result[0]
    sizes_arr = result[1]

    set_labels = []
    for ps in sets_list:
        labels = [CLASS_NAMES_S2[i] for i in ps]
        set_labels.append(labels)

    cascade[f'{key_prefix}_conformal_set'] = set_labels
    cascade[f'{key_prefix}_conformal_size'] = sizes_arr

    n_total = len(cascade)
    coverage = sum(1 for i, ps in enumerate(sets_list)
                   if cascade.iloc[i]['true_label'] in ps) / n_total
    singleton = sum(1 for s in sizes_arr if s == 1) / n_total
    avg_size = np.mean(sizes_arr)

    print(f"Conformal ({scenario}, α=0.10, q_hat={q_hat:.4f}):")
    print(f"  Coverage:  {coverage:.3f}  (target ≥0.90)")
    print(f"  Singleton: {singleton:.3f}")
    print(f"  Avg size:  {avg_size:.2f}")

# Re-save with conformal columns
cascade.to_parquet(M11_DATA / 'temporal_cascade_simulation.parquet', index=False)
print(f"\n✅ Cascade + conformal saved")

In [ ]:
##Cell 6 — Compute All Temporal Metrics (Phase 3)

In [ ]:
# ═══════════════════════════════════════════════════════════
# M11 · Cell 6: Temporal Metrics — All Phase 3 Computations
# ═══════════════════════════════════════════════════════════
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score, brier_score_loss,
                             confusion_matrix, classification_report)
from sklearn.calibration import calibration_curve

n_temporal = len(cascade)

# ═══════════════════════════════════════════════════════════
# 1. BASELINE: S2 stage-level metrics (both scenarios)
# ═══════════════════════════════════════════════════════════
print("1. STAGE-LEVEL BASELINE METRICS (S2)")
print("=" * 60)

baseline_rows = []
for scenario in ['CBC_Only', 'CBC_BIO']:
    df = predictions[f'S2_{scenario}']['df']
    y_true = df['target'].values
    y_pred = df['pred'].values
    probs = df[[f'prob_{c}_cal' for c in CLASS_NAMES_S2]].values

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='macro', zero_division=0)

    # ROC AUC (one-vs-rest)
    try:
        auc = roc_auc_score(y_true, probs, multi_class='ovr', average='macro')
    except:
        auc = np.nan

    baseline_rows.append({
        'Scenario': scenario, 'Stage': 'Stage 2', 'Split': 'Temporal',
        'N': len(df), 'Accuracy': round(acc, 3),
        'Macro Precision': round(prec, 3), 'Macro Recall': round(rec, 3),
        'Macro F1': round(f1, 3), 'ROC AUC': round(auc, 3)
    })
    print(f"  {scenario}: Acc={acc:.3f}, F1={f1:.3f}, AUC={auc:.3f}")

# E2E (cascade)
cascade_acc = cascade['correct'].mean()
cascade_f1 = f1_score(cascade['true_label'], cascade['final_pred'], average='macro')
baseline_rows.append({
    'Scenario': 'Cascade', 'Stage': 'E2E', 'Split': 'Temporal',
    'N': n_temporal, 'Accuracy': round(cascade_acc, 3),
    'Macro Precision': np.nan, 'Macro Recall': np.nan,
    'Macro F1': round(cascade_f1, 3), 'ROC AUC': np.nan
})
print(f"  Cascade E2E: Acc={cascade_acc:.3f}, F1={cascade_f1:.3f}")

temporal_baseline = pd.DataFrame(baseline_rows)

# ═══════════════════════════════════════════════════════════
# 2. PER-CLASS METRICS
# ═══════════════════════════════════════════════════════════
print(f"\n2. PER-CLASS METRICS")
print("=" * 60)

perclass_rows = []
for scenario in ['CBC_Only', 'CBC_BIO']:
    df = predictions[f'S2_{scenario}']['df']
    y_true = df['target'].values
    y_pred = df['pred'].values

    for ci, cname in enumerate(CLASS_NAMES_S2):
        mask = y_true == ci
        n_cls = mask.sum()
        tp = ((y_pred == ci) & (y_true == ci)).sum()
        fp = ((y_pred == ci) & (y_true != ci)).sum()
        fn = ((y_pred != ci) & (y_true == ci)).sum()

        p = tp / (tp + fp) if (tp + fp) > 0 else 0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1c = 2 * p * r / (p + r) if (p + r) > 0 else 0

        perclass_rows.append({
            'Scenario': scenario, 'Stage': 'Stage 2', 'Split': 'Temporal',
            'Class': {'DEA': 'IDA', 'HGB_HTZ': 'HGB HTZ'}.get(CLASS_LABELS[ci], CLASS_LABELS[ci]), 'N': n_cls,
            'Precision': round(p, 3), 'Recall': round(r, 3), 'F1': round(f1c, 3)
        })

    # MACRO AVG
    prec_m = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec_m = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1_m = f1_score(y_true, y_pred, average='macro')
    perclass_rows.append({
        'Scenario': scenario, 'Stage': 'Stage 2', 'Split': 'Temporal',
        'Class': 'MACRO AVG', 'N': len(df),
        'Precision': round(prec_m, 3), 'Recall': round(rec_m, 3), 'F1': round(f1_m, 3)
    })

temporal_perclass = pd.DataFrame(perclass_rows)
print(temporal_perclass.to_string(index=False))

# ═══════════════════════════════════════════════════════════
# 3. CALIBRATION METRICS
# ═══════════════════════════════════════════════════════════
print(f"\n3. CALIBRATION METRICS")
print("=" * 60)

cal_rows = []
for scenario in ['CBC_Only', 'CBC_BIO']:
    df = predictions[f'S2_{scenario}']['df']
    y_true = df['target'].values
    probs = df[[f'prob_{c}_cal' for c in CLASS_NAMES_S2]].values

    # Brier score (multiclass: mean of per-class Brier)
    brier = 0
    for ci in range(4):
        y_bin = (y_true == ci).astype(float)
        brier += brier_score_loss(y_bin, probs[:, ci])
    brier /= 4

    # ECE (equal-width bins, per-class averaged)
    ece_sum = 0
    n_bins = 10
    for ci in range(4):
        y_bin = (y_true == ci).astype(float)
        p = probs[:, ci]
        for b in range(n_bins):
            lo, hi = b / n_bins, (b + 1) / n_bins
            mask = (p >= lo) & (p < hi)
            if mask.sum() > 0:
                ece_sum += mask.sum() * abs(y_bin[mask].mean() - p[mask].mean())
    ece = ece_sum / (len(y_true) * 4)

    cal_rows.append({
        'Scenario': scenario, 'Stage': 'Stage 2', 'Split': 'Temporal',
        'Brier Score': round(brier, 4), 'ECE': round(ece, 4)
    })
    print(f"  {scenario}: Brier={brier:.4f}, ECE={ece:.4f}")

temporal_calibration = pd.DataFrame(cal_rows)

# ═══════════════════════════════════════════════════════════
# 4. CONFORMAL METRICS (multiple α levels)
# ═══════════════════════════════════════════════════════════
print(f"\n4. CONFORMAL METRICS")
print("=" * 60)

conformal_rows = []
for scenario in ['CBC_Only', 'CBC_BIO']:
    s2_df = predictions[f'S2_{scenario}']['df']
    probs_cal = s2_df[[f'prob_{c}_cal' for c in CLASS_NAMES_S2]].values
    y_true = s2_df['target'].values

    for alpha in [0.05, 0.10, 0.20]:
        q_key = f"{scenario}_S2_a{alpha}"
        q_hat = Q_HATS[q_key]

        result = predict_sets(probs_cal, q_hat, randomized=True,
                              rng=np.random.default_rng(42))
        sets_list, sizes_arr = result

        coverage = sum(1 for i, ps in enumerate(sets_list) if y_true[i] in ps) / len(y_true)
        singleton = sum(1 for s in sizes_arr if s == 1) / len(y_true)
        avg_size = np.mean(sizes_arr)

        conformal_rows.append({
            'Scenario': scenario, 'Stage': 2, 'Alpha': alpha,
            'Split': 'Temporal', 'q_hat': round(q_hat, 6),
            'Coverage': round(coverage, 3), 'Singleton Rate': round(singleton, 3),
            'Avg Set Size': round(avg_size, 2), 'N': len(y_true)
        })

        if alpha == 0.10:
            print(f"  {scenario} α={alpha}: cov={coverage:.3f}, sing={singleton:.3f}, size={avg_size:.2f}")

temporal_conformal = pd.DataFrame(conformal_rows)

# ═══════════════════════════════════════════════════════════
# 5. CASCADE METRICS
# ═══════════════════════════════════════════════════════════
print(f"\n5. CASCADE METRICS")
print("=" * 60)

n_esc = cascade['escalated'].sum()
n_tier1 = n_temporal - n_esc
t1_high = cascade[(cascade['final_tier'] == 1) & (cascade['final_zone'] == 'HIGH')]

cascade_metrics_dict = {
    'Cascade Accuracy': round(cascade_acc, 3),
    'Cascade Macro F1': round(cascade_f1, 3),
    'Total Patients': n_temporal,
    'Tier 1 Retained': n_tier1,
    'Tier 1 Retained (%)': round(n_tier1 / n_temporal * 100, 1),
    'Tier 1 Accuracy': round(cascade[cascade['final_tier'] == 1]['correct'].mean(), 3),
    'Tier 1 HIGH n': len(t1_high),
    'Tier 1 HIGH Accuracy': round(t1_high['correct'].mean(), 3) if len(t1_high) > 0 else np.nan,
    'Escalated': n_esc,
    'Escalation Rate (%)': round(n_esc / n_temporal * 100, 1),
    'Tier 2 Accuracy': round(cascade[cascade['final_tier'] == 2]['correct'].mean(), 3),
    'Bio Utilization (%)': round(n_esc / n_temporal * 100, 1),
}

temporal_cascade_metrics = pd.DataFrame([
    {'Metric': k, 'Value': v} for k, v in cascade_metrics_dict.items()
])
print(temporal_cascade_metrics.to_string(index=False))

# ═══════════════════════════════════════════════════════════
# 6. SAVE ALL METRICS
# ═══════════════════════════════════════════════════════════
with pd.ExcelWriter(M11_ANALYSIS / 'temporal_all_metrics.xlsx') as writer:
    temporal_baseline.to_excel(writer, sheet_name='Baseline', index=False)
    temporal_perclass.to_excel(writer, sheet_name='PerClass', index=False)
    temporal_calibration.to_excel(writer, sheet_name='Calibration', index=False)
    temporal_conformal.to_excel(writer, sheet_name='Conformal', index=False)
    temporal_cascade_metrics.to_excel(writer, sheet_name='Cascade', index=False)

print(f"\n✅ All temporal metrics saved → temporal_all_metrics.xlsx")

## Cell 7 — Phase 4: Distribution Shift Detection

In [ ]:
# ═══════════════════════════════════════════════════════════
# M11 · Cell 7: Distribution Shift Detection (Train vs Temporal)
# ═══════════════════════════════════════════════════════════
from scipy import stats

# ── Source: S1 BIO OOF as "training" reference (AAC patients only) ──
train_ref = s1_bio_oof[s1_bio_oof['target'] == 1].copy()
temporal_data = pd.read_parquet(M11_DATA / 'temporal_cohort.parquet')

# All base features to compare
base_features_all = ['mcv_f_l', 'ret_he_pg', 'rbc_10_6_u_l', 'ret_number_10_6_l',
                     'rdw_sd_fl', 'delta_he_pg', 'frc_perc', 'mchc_g_dl',
                     'irf_pct', 'nrbc_pct', 'hgb_g_d_l', 'micro_macro_ratio',
                     'ferritin', 'demir', 'ldh', 'uibc', 'yas']

# ═══════════════════════════════════════════════════════════
# 1. Per-feature KS test + PSI
# ═══════════════════════════════════════════════════════════

def compute_psi(train, test, bins=10):
    """Population Stability Index."""
    eps = 1e-4
    # Use train quantiles as bin edges
    breakpoints = np.quantile(train.dropna(), np.linspace(0, 1, bins + 1))
    breakpoints = np.unique(breakpoints)
    if len(breakpoints) < 3:
        return np.nan

    train_counts = np.histogram(train.dropna(), bins=breakpoints)[0]
    test_counts = np.histogram(test.dropna(), bins=breakpoints)[0]

    train_pct = train_counts / train_counts.sum() + eps
    test_pct = test_counts / test_counts.sum() + eps

    psi = np.sum((test_pct - train_pct) * np.log(test_pct / train_pct))
    return psi

shift_rows = []
for feat in base_features_all:
    train_vals = train_ref[feat].dropna()
    temp_vals = temporal_data[feat].dropna()

    # KS test
    ks_stat, ks_p = stats.ks_2samp(train_vals, temp_vals)

    # PSI
    psi = compute_psi(train_vals, temp_vals)

    # Mean shift
    train_mean = train_vals.mean()
    temp_mean = temp_vals.mean()
    pct_shift = (temp_mean - train_mean) / train_mean * 100 if train_mean != 0 else 0

    # Verdict
    if psi > 0.2 or ks_p < 0.001:
        verdict = '❌ Significant'
    elif psi > 0.1 or ks_p < 0.05:
        verdict = '⚠️ Minor drift'
    else:
        verdict = '✅ Stable'

    shift_rows.append({
        'Feature': feat,
        'Train Mean': round(train_mean, 3),
        'Temporal Mean': round(temp_mean, 3),
        'Shift (%)': round(pct_shift, 1),
        'KS Statistic': round(ks_stat, 4),
        'KS p-value': round(ks_p, 4),
        'PSI': round(psi, 4) if not np.isnan(psi) else np.nan,
        'Verdict': verdict,
    })

shift_df = pd.DataFrame(shift_rows).sort_values('PSI', ascending=False)

print("Feature Distribution Shift: Train (OOF) vs Temporal")
print("=" * 110)
print(shift_df.to_string(index=False))

n_sig = (shift_df['Verdict'] == '❌ Significant').sum()
n_minor = (shift_df['Verdict'].str.contains('Minor')).sum()
n_stable = (shift_df['Verdict'] == '✅ Stable').sum()
print(f"\nSummary: {n_stable} stable, {n_minor} minor drift, {n_sig} significant")

# ═══════════════════════════════════════════════════════════
# 2. Class distribution comparison (chi-square)
# ═══════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("Class Distribution: Chi-Square Test")
print("=" * 60)

train_class_dist = train_ref.merge(
    s2_bio_oof[['target']].reset_index(drop=True),
    left_index=True, right_index=True, suffixes=('_s1', '')
)['target'].value_counts().sort_index()

# Normalize to temporal sample size
expected = (train_class_dist / train_class_dist.sum() * n_temporal).values
observed = temporal_data['target'].value_counts().sort_index().values

chi2, chi_p = stats.chisquare(observed, f_exp=expected)
_class_disp = ['IDA', 'HA', 'HGB HTZ', 'Normal']  # display labels for printing
print(f"  Train dist (normalized to n={n_temporal}): {dict(zip(_class_disp, expected.round(1)))}")
print(f"  Temporal dist:  {dict(zip(_class_disp, observed))}")
print(f"  Chi-square: {chi2:.3f}, p={chi_p:.4f}")
print(f"  {'⚠️ Significant shift' if chi_p < 0.05 else '✅ No significant shift'}")

# ═══════════════════════════════════════════════════════════
# 3. Save
# ═══════════════════════════════════════════════════════════
shift_df.to_excel(M11_ANALYSIS / 'temporal_feature_shift.xlsx', index=False)
print(f"\n✅ Saved → temporal_feature_shift.xlsx")

# Store for figures later
top10_shifted = shift_df.head(10)['Feature'].tolist()
print(f"Top 10 shifted features (for Fig S6): {top10_shifted}")

## Phase 6: Test vs Temporal Comparison Table

In [ ]:
# ═══════════════════════════════════════════════════════════
# M11 · Cell 8: Test vs Temporal Comparison (Delta Table)
# ═══════════════════════════════════════════════════════════

# ── Load original test metrics ──
orig_tables = pd.ExcelFile(TAB_DIR / 'ALL_PUBLICATION_TABLES.xlsx')

# Table 1: Baseline
t1 = pd.read_excel(orig_tables, sheet_name='Table_1_Baseline')

# Table 6A: Conformal
t6a = pd.read_excel(orig_tables, sheet_name='Table_6A_Conformal')

# Table 7A: Cascade
t7a = pd.read_excel(orig_tables, sheet_name='Table_7A_Cascade')

# ── Build comparison rows ──
comp_rows = []

def add_row(metric, test_val, temp_val, threshold=0.03):
    delta = temp_val - test_val
    pct = delta / test_val * 100 if test_val != 0 else 0
    if abs(delta) < 0.01:
        verdict = '✅ Stable'
    elif abs(delta) < threshold:
        verdict = '⚠️ Minor drift'
    else:
        verdict = '❌ Significant'
    comp_rows.append({
        'Metric': metric,
        'Test': round(test_val, 3),
        'Temporal': round(temp_val, 3),
        'Δ': round(delta, 3),
        'Δ (%)': round(pct, 1),
        'Verdict': verdict,
    })

# S2 CBC_Only
s2_cbc_test_acc = t1[
    (t1['Stage'] == 'Stage 2') &
    (t1['Scenario'].str.contains('CBC Only'))
]['Accuracy'].values[0]
s2_cbc_temp = temporal_baseline[
    (temporal_baseline['Scenario'] == 'CBC_Only') &
    (temporal_baseline['Stage'] == 'Stage 2')
]
add_row('S2 CBC Only — Accuracy', s2_cbc_test_acc, s2_cbc_temp['Accuracy'].values[0])
add_row('S2 CBC Only — Macro F1',
        t1[(t1['Stage'] == 'Stage 2') & (t1['Scenario'].str.contains('CBC Only'))]['Macro F1'].values[0],
        s2_cbc_temp['Macro F1'].values[0])

# S2 CBC_BIO
s2_bio_test_acc = t1[
    (t1['Stage'] == 'Stage 2') &
    (t1['Scenario'].str.contains('CBC.*Bio', case=False, regex=True))
]['Accuracy'].values[0]
s2_bio_temp = temporal_baseline[
    (temporal_baseline['Scenario'] == 'CBC_BIO') &
    (temporal_baseline['Stage'] == 'Stage 2')
]
add_row('S2 CBC+BIO — Accuracy', s2_bio_test_acc, s2_bio_temp['Accuracy'].values[0])
add_row('S2 CBC+BIO — Macro F1',
        t1[(t1['Stage'] == 'Stage 2') & (t1['Scenario'].str.contains('CBC.*Bio', case=False, regex=True))]['Macro F1'].values[0],
        s2_bio_temp['Macro F1'].values[0])

# E2E / Cascade
e2e_test_acc = t1[t1['Stage'] == 'E2E']
if not e2e_test_acc.empty:
    # Use CBC Only E2E as baseline (cascade is CBC_Only-based)
    e2e_val = e2e_test_acc[e2e_test_acc['Scenario'].str.contains('CBC Only')]['Accuracy'].values[0]
    add_row('E2E CBC Only — Accuracy', e2e_val,
            temporal_baseline[temporal_baseline['Stage'] == 'E2E']['Accuracy'].values[0])

# Cascade-specific
cascade_test_acc = t7a[t7a['Metric'] == 'Cascade Accuracy']['Value'].values
if len(cascade_test_acc) > 0:
    add_row('Cascade — Accuracy', float(cascade_test_acc[0]),
            cascade_metrics_dict['Cascade Accuracy'])

# Conformal coverage (α=0.10, CBC_BIO)
conf_test_bio = t6a[
    (t6a['Scenario'].str.contains('CBC.*Bio', case=False, regex=True)) &
    (t6a['α'] == 0.10)
]
conf_temp_bio = temporal_conformal[
    (temporal_conformal['Scenario'] == 'CBC_BIO') &
    (temporal_conformal['Alpha'] == 0.10)
]
if not conf_test_bio.empty and not conf_temp_bio.empty:
    add_row('Conformal CBC+BIO α=0.10 — Coverage',
            conf_test_bio['Empirical Coverage'].values[0],
            conf_temp_bio['Coverage'].values[0], threshold=0.05)
    add_row('Conformal CBC+BIO α=0.10 — Singleton',
            conf_test_bio['Singleton Rate'].values[0],
            conf_temp_bio['Singleton Rate'].values[0], threshold=0.10)

# Conformal coverage (α=0.10, CBC_Only)
conf_test_cbc = t6a[
    (t6a['Scenario'].str.contains('CBC Only')) &
    (t6a['α'] == 0.10)
]
conf_temp_cbc = temporal_conformal[
    (temporal_conformal['Scenario'] == 'CBC_Only') &
    (temporal_conformal['Alpha'] == 0.10)
]
if not conf_test_cbc.empty and not conf_temp_cbc.empty:
    add_row('Conformal CBC Only α=0.10 — Coverage',
            conf_test_cbc['Empirical Coverage'].values[0],
            conf_temp_cbc['Coverage'].values[0], threshold=0.05)

comparison = pd.DataFrame(comp_rows)

print("TEST vs TEMPORAL — Key Metrics Comparison")
print("=" * 90)
print(comparison.to_string(index=False))

n_stable = (comparison['Verdict'] == '✅ Stable').sum()
n_minor = (comparison['Verdict'].str.contains('Minor')).sum()
n_sig = (comparison['Verdict'].str.contains('Significant')).sum()
print(f"\n  Summary: {n_stable} stable, {n_minor} minor, {n_sig} significant")

# ── Save ──
comparison.to_excel(M11_ANALYSIS / 'temporal_vs_test_comparison.xlsx', index=False)
print(f"\n✅ Saved → temporal_vs_test_comparison.xlsx")

## Cell 9 — Figure Setup

In [ ]:
# ═══════════════════════════════════════════════════════════
# M11 · Cell 9: Figure Setup + Fig S7 Confusion Matrix
# ═══════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap

mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']
CHOSEN_FONT = 'DejaVu Sans'
import plot_style
plot_style.CHOSEN_FONT = 'DejaVu Sans'

SINGLE_COL = 84 / 25.4
DOUBLE_COL = 174 / 25.4

C_HIGH   = PALETTE['highlight']
C_BASE   = PALETTE['base1']
C_GREY   = '#95A5A6'
C_GREEN  = PALETTE['accent1']
C_ORANGE = PALETTE['accent2']
C_PURPLE = PALETTE['accent3']
SC_COLORS = {'CBC_Only': C_BASE, 'CBC_BIO': C_PURPLE}
SC_LABELS = {'CBC_Only': 'CBC Only', 'CBC_BIO': 'CBC + Biochemistry'}
TEMPORAL_COLOR = '#2C3E50'
TEMPORAL_MARKER = 'D'
CL_COLORS = CLASS_COLORS  # from cds_config
cmap_tufte = LinearSegmentedColormap.from_list('tufte', ['#FFFFFF', C_BASE], N=256)

print("✅ Figure setup complete")

## Fig S7: Temporal Confusion Matrix

In [ ]:
# ═══════════════════════════════════════════════════════════
# Fig S7: Temporal Normalized Confusion Matrices (S2, side-by-side)
# ═══════════════════════════════════════════════════════════
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(DOUBLE_COL, DOUBLE_COL * 0.42))
class_display = ['IDA', 'HA', 'HGB HTZ', 'Normal']

for idx, scenario in enumerate(['CBC_Only', 'CBC_BIO']):
    ax = axes[idx]
    df = predictions[f'S2_{scenario}']['df']
    y_true = df['target'].values
    y_pred = df['pred'].values

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    im = ax.imshow(cm_norm, cmap=cmap_tufte, vmin=0, vmax=1, aspect='equal')

    for i in range(4):
        for j in range(4):
            val = cm_norm[i, j]
            count = cm[i, j]
            color = 'white' if val > 0.5 else '#333333'
            ax.text(j, i, f'{val:.2f}\n({count})',
                    ha='center', va='center', fontsize=7, color=color)

    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels(class_display, fontsize=7, rotation=45, ha='right')
    ax.set_yticklabels(class_display, fontsize=7)
    ax.set_xlabel('Predicted', fontsize=8)
    ax.set_ylabel('Actual', fontsize=8)

    title = f'S2 {SC_LABELS[scenario]}'
    ax.set_title(title, fontsize=9, pad=8)
    add_panel_label(ax, ['A', 'B'][idx], fontsize=14)

fig.suptitle('Temporal Cohort — Normalized Confusion Matrices (S2)',
             fontsize=10, fontweight='bold', y=1.02)
fig.tight_layout(w_pad=3.0)
save_fig(fig, FIG_DIR, 'supp_s7_temporal_confusion_matrix')
plt.show()
print("✅ Fig S7 saved")

## # M11 · Cell 11: Fig S6 — Feature Distribution Shift (Violin)

In [ ]:
# ═══════════════════════════════════════════════════════════
# M11 · Cell 11: Fig S6 — Feature Distribution Shift (Violin)
# ═══════════════════════════════════════════════════════════

train_ref_ias = s1_bio_oof[s1_bio_oof['target'] == 1].copy()
temporal_data = pd.read_parquet(M11_DATA / 'temporal_cohort.parquet')

# Top 10 shifted features from Cell 7
top10 = ['yas', 'mchc_g_dl', 'ldh', 'hgb_g_d_l', 'uibc',
         'ret_he_pg', 'frc_perc', 'rdw_sd_fl', 'ret_number_10_6_l', 'mcv_f_l']

# Display names
FEAT_DISPLAY = {
    'yas': 'Age', 'mchc_g_dl': 'MCHC', 'ldh': 'LD', 'hgb_g_d_l': 'HGB',
    'uibc': 'UIBC', 'ret_he_pg': 'Ret-He', 'frc_perc': 'FRC%',
    'rdw_sd_fl': 'RDW-SD', 'ret_number_10_6_l': 'RET#', 'mcv_f_l': 'MCV'
}

fig, axes = plt.subplots(2, 5, figsize=(DOUBLE_COL, DOUBLE_COL * 0.52))
axes = axes.flatten()

for i, feat in enumerate(top10):
    ax = axes[i]

    train_vals = train_ref_ias[feat].dropna().values
    temp_vals = temporal_data[feat].dropna().values

    # Z-score normalize for comparable violin widths
    combined = np.concatenate([train_vals, temp_vals])
    mu, sigma = combined.mean(), combined.std()
    if sigma == 0:
        sigma = 1

    parts_t = ax.violinplot([train_vals], positions=[0], showmedians=True,
                            showextrema=False, widths=0.7)
    parts_p = ax.violinplot([temp_vals], positions=[1], showmedians=True,
                            showextrema=False, widths=0.7)

    # Color: train=grey, temporal=charcoal
    for pc in parts_t['bodies']:
        pc.set_facecolor(C_GREY)
        pc.set_alpha(0.6)
    parts_t['cmedians'].set_color(C_GREY)
    parts_t['cmedians'].set_linewidth(1.5)

    for pc in parts_p['bodies']:
        pc.set_facecolor(C_PURPLE)
        pc.set_alpha(0.6)
    parts_p['cmedians'].set_color(TEMPORAL_COLOR)
    parts_p['cmedians'].set_linewidth(1.5)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Train', 'Temp.'], fontsize=6)
    ax.set_title(FEAT_DISPLAY.get(feat, feat), fontsize=7.5, fontweight='bold')
    ax.tick_params(labelsize=6)

    # PSI annotation
    shift_row = shift_df[shift_df['Feature'] == feat]
    if not shift_row.empty:
        psi = shift_row['PSI'].values[0]
        verdict = shift_row['Verdict'].values[0]
        color = C_HIGH if '❌' in verdict else (C_ORANGE if '⚠️' in verdict else C_GREEN)
        psi_text = f'PSI={psi:.2f}' if not np.isnan(psi) else 'PSI=N/A'
        ax.text(0.5, 0.97, psi_text, transform=ax.transAxes, ha='center', va='top',
                fontsize=5.5, color=color, fontweight='bold')

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Legend
legend_items = [
    mpatches.Patch(facecolor=C_GREY, alpha=0.6, label='Training (OOF)'),
    mpatches.Patch(facecolor=C_PURPLE, alpha=0.6, label='Temporal'),
]
fig.legend(handles=legend_items, loc='lower center', ncol=2,
           fontsize=7, bbox_to_anchor=(0.5, -0.02))

fig.suptitle('Feature Distribution Shift — Top 10 Features (Train vs Temporal)',
             fontsize=9.5, fontweight='bold', y=1.01)
fig.tight_layout(h_pad=2.0, w_pad=1.5)
save_fig(fig, FIG_DIR, 'supp_s6_feature_distribution_shift')
plt.show()
print("✅ Fig S6 saved")

## M11 · Cell 12: Fig 2 UPDATED — Calibration + Temporal Overlay

In [ ]:
# ═══════════════════════════════════════════════════════════
# M11 · Cell 12: Fig 2 UPDATED — Calibration + Temporal Overlay
# ═══════════════════════════════════════════════════════════
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(2, 2, figsize=(DOUBLE_COL, DOUBLE_COL * 0.85))

panels = [
    ('1', 'CBC_Only', 'S1 — CBC Only'),
    ('1', 'CBC_BIO',  'S1 — CBC + Biochemistry'),
    ('2', 'CBC_Only', 'S2 — CBC Only'),
    ('2', 'CBC_BIO',  'S2 — CBC + Biochemistry'),
]
panel_labels = ['A', 'B', 'C', 'D']

for idx, (stage, scenario, title) in enumerate(panels):
    ax = axes[idx // 2, idx % 2]
    df = prob[f"S{stage}_{scenario}_test"]

    if stage == '1':
        # ── Binary: AAC probability (UNCHANGED from M9) ──
        y_true = df['target']
        prob_uncal = df['prob_IAS']
        prob_cal = df['prob_IAS_cal']

        frac_pos_u, mean_pred_u = calibration_curve(y_true, prob_uncal, n_bins=8, strategy='uniform')
        frac_pos_c, mean_pred_c = calibration_curve(y_true, prob_cal, n_bins=8, strategy='uniform')

        ax.plot([0, 1], [0, 1], '--', color=C_GREY, lw=0.8, zorder=0)
        ax.plot(mean_pred_u, frac_pos_u, 'o-', color=C_GREY, lw=1.2,
                markersize=4, label='Uncalibrated')
        ax.plot(mean_pred_c, frac_pos_c, 's-', color=SC_COLORS[scenario], lw=1.2,
                markersize=4, label='Calibrated')
        # S1: no temporal overlay (all temporal patients are IAS → single-class)

    else:
        # ── Multiclass: per-class one-vs-rest (UNCHANGED from M9) ──
        class_names = ['DEA', 'HA', 'HGB_HTZ', 'Normal']
        class_display = ['IDA', 'HA', 'HGB HTZ', 'Normal']
        colors = [CL_COLORS.get(cn, C_BASE) for cn in class_names]  # color lookup by internal key

        ax.plot([0, 1], [0, 1], '--', color=C_GREY, lw=0.8, zorder=0)

        for ci, cname in enumerate(class_names):
            y_bin = (df['target'] == ci).astype(int)
            prob_col = f'prob_{cname}_cal' if f'prob_{cname}_cal' in df.columns else f'prob_{cname}'
            p = df[prob_col]
            try:
                frac_pos, mean_pred = calibration_curve(y_bin, p, n_bins=6, strategy='uniform')
                ax.plot(mean_pred, frac_pos, 'o-', color=colors[ci], lw=1.0,
                        markersize=3.5, label=class_display[ci])
            except:
                pass

        # ── TEMPORAL OVERLAY (NEW) ──
        temp_df = predictions[f'S2_{scenario}']['df']
        for ci, cname in enumerate(class_names):
            y_bin_t = (temp_df['target'] == ci).astype(int)
            prob_col_t = f'prob_{cname}_cal'
            p_t = temp_df[prob_col_t]
            try:
                frac_pos_t, mean_pred_t = calibration_curve(y_bin_t, p_t, n_bins=6, strategy='uniform')
                ax.plot(mean_pred_t, frac_pos_t, TEMPORAL_MARKER,
                        color=colors[ci], markersize=4, alpha=0.7,
                        markeredgecolor=TEMPORAL_COLOR, markeredgewidth=0.8,
                        zorder=6)
            except:
                pass

    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_xlabel('Predicted probability', fontsize=8)
    ax.set_ylabel('Observed frequency', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=6, loc='lower right')
    add_panel_label(ax, panel_labels[idx], fontsize=14)
    ax.set_title(title, fontsize=8.5, pad=6)

# Add temporal legend entry to S2 panels only
from matplotlib.lines import Line2D
temporal_handle = Line2D([0], [0], marker=TEMPORAL_MARKER, color='none',
                         markeredgecolor=TEMPORAL_COLOR, markeredgewidth=0.8,
                         markersize=5, label='Temporal')
for ax_idx in [2, 3]:  # panels C, D
    ax = axes[ax_idx // 2, ax_idx % 2]
    handles, labels = ax.get_legend_handles_labels()
    handles.append(temporal_handle)
    labels.append('Temporal')
    ax.legend(handles, labels, fontsize=6, loc='lower right')

fig.tight_layout(h_pad=2.5, w_pad=2.0)
save_fig(fig, FIG_DIR, 'fig2_calibration_reliability')
plt.show()
print("✅ Fig 2 updated with temporal overlay")

## Cell 13 — Fig 3 Updated: Conformal + Temporal Overlay

In [ ]:
# ═══════════════════════════════════════════════════════════
# M11 · Cell 13: Fig 3 UPDATED — Conformal + Temporal Overlay
# ═══════════════════════════════════════════════════════════

# ── Original test conformal summary ──
cs = conf_summary[conf_summary['method'] == 'randomized'].copy()
alphas = [0.05, 0.10, 0.20]

fig, axes = plt.subplots(1, 3, figsize=(DOUBLE_COL, DOUBLE_COL * 0.38))

# ── Panel A: Coverage (UNCHANGED + temporal) ──
ax = axes[0]
for sc in ['CBC_Only', 'CBC_BIO']:
    sub = cs[cs['scenario'].str.strip() == sc]
    ax.plot(sub['alpha'], sub['empirical_coverage'], 'o-',
            color=SC_COLORS[sc], lw=1.3, markersize=5, label=SC_LABELS[sc])
targets = [0.95, 0.90, 0.80]
ax.plot(alphas, targets, '--', color=C_GREY, lw=0.8, label='Target (1−α)')

# TEMPORAL overlay
for sc in ['CBC_Only', 'CBC_BIO']:
    temp_covs = []
    for alpha in alphas:
        row = temporal_conformal[(temporal_conformal['Scenario'] == sc) &
                                 (temporal_conformal['Alpha'] == alpha)]
        temp_covs.append(row['Coverage'].values[0])
    ax.plot(alphas, temp_covs, TEMPORAL_MARKER,
            color=SC_COLORS[sc], markersize=6,
            markeredgecolor=TEMPORAL_COLOR, markeredgewidth=1.0, zorder=6)

ax.set_xlabel('α (miscoverage level)', fontsize=8)
ax.set_ylabel('Empirical coverage', fontsize=8)
ax.set_xticks(alphas)
ax.set_ylim(0.75, 1.02)
ax.tick_params(labelsize=7)
ax.legend(fontsize=6, loc='lower left')
add_panel_label(ax, 'A', fontsize=14)

# ── Panel B: Set size (UNCHANGED + temporal) ──
ax = axes[1]
for sc in ['CBC_Only', 'CBC_BIO']:
    sub = cs[cs['scenario'].str.strip() == sc]
    ax.plot(sub['alpha'], sub['avg_set_size'], 's-',
            color=SC_COLORS[sc], lw=1.3, markersize=5, label=SC_LABELS[sc])
ax.axhline(y=4, color=C_GREY, ls=':', lw=0.6, alpha=0.5)
ax.axhline(y=1, color=C_GREEN, ls=':', lw=0.6, alpha=0.5)
ax.text(0.205, 1.12, 'singleton', fontsize=5.5, color=C_GREEN, ha='right')
ax.text(0.205, 4.12, 'all classes', fontsize=5.5, color=C_GREY, ha='right')

# TEMPORAL overlay
for sc in ['CBC_Only', 'CBC_BIO']:
    temp_sizes = []
    for alpha in alphas:
        row = temporal_conformal[(temporal_conformal['Scenario'] == sc) &
                                 (temporal_conformal['Alpha'] == alpha)]
        temp_sizes.append(row['Avg Set Size'].values[0])
    ax.plot(alphas, temp_sizes, TEMPORAL_MARKER,
            color=SC_COLORS[sc], markersize=6,
            markeredgecolor=TEMPORAL_COLOR, markeredgewidth=1.0, zorder=6)

ax.set_xlabel('α (miscoverage level)', fontsize=8)
ax.set_ylabel('Average set size', fontsize=8)
ax.set_xticks(alphas)
ax.set_ylim(0.5, 4.5)
ax.tick_params(labelsize=7)
ax.legend(fontsize=6, loc='upper right')
add_panel_label(ax, 'B', fontsize=14)

# ── Panel C: Singleton rate (UNCHANGED + temporal) ──
ax = axes[2]
for sc in ['CBC_Only', 'CBC_BIO']:
    sub = cs[cs['scenario'].str.strip() == sc]
    ax.plot(sub['alpha'], sub['singleton_rate'] * 100, 'D-',
            color=SC_COLORS[sc], lw=1.3, markersize=5, label=SC_LABELS[sc])

# Original annotations (same as M9)
for sc, xytext_offset in [('CBC_BIO', (0.035, 3)), ('CBC_Only', (0.035, 3))]:
    sub = cs[(cs['scenario'].str.strip() == sc) & (cs['alpha'] == 0.10)]
    if not sub.empty:
        sr = sub['singleton_rate'].values[0] * 100
        ax.annotate(f'{sr:.0f}%', xy=(0.10, sr),
                    xytext=(0.10 + xytext_offset[0], sr + xytext_offset[1]),
                    fontsize=6.5, color=SC_COLORS[sc], fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color=SC_COLORS[sc], lw=0.6))

# TEMPORAL overlay
for sc in ['CBC_Only', 'CBC_BIO']:
    temp_sings = []
    for alpha in alphas:
        row = temporal_conformal[(temporal_conformal['Scenario'] == sc) &
                                 (temporal_conformal['Alpha'] == alpha)]
        temp_sings.append(row['Singleton Rate'].values[0] * 100)
    ax.plot(alphas, temp_sings, TEMPORAL_MARKER,
            color=SC_COLORS[sc], markersize=6,
            markeredgecolor=TEMPORAL_COLOR, markeredgewidth=1.0, zorder=6)

ax.set_xlabel('α (miscoverage level)', fontsize=8)
ax.set_ylabel('Singleton rate (%)', fontsize=8)
ax.set_xticks(alphas)
ax.set_ylim(-3, 68)
ax.tick_params(labelsize=7)
ax.legend(fontsize=6, loc='upper left')
add_panel_label(ax, 'C', fontsize=14)

# Add temporal legend to all panels
temporal_handle = Line2D([0], [0], marker=TEMPORAL_MARKER, color='none',
                         markeredgecolor=TEMPORAL_COLOR, markeredgewidth=1.0,
                         markersize=5, label='Temporal')
panel_locs = ['lower left', 'upper right', 'upper left']
for i, ax in enumerate(axes):
    handles, labels = ax.get_legend_handles_labels()
    handles.append(temporal_handle)
    labels.append('Temporal')
    ax.legend(handles, labels, fontsize=6, loc=panel_locs[i])

fig.tight_layout(w_pad=2.5)
save_fig(fig, FIG_DIR, 'fig3_conformal_prediction')
plt.show()
print("✅ Fig 3 updated with temporal overlay")

## Cell 14 — Fig 4 Updated: Efficiency Curve + Temporal Point

In [ ]:
# ═══════════════════════════════════════════════════════════
# M11 · Cell 14: Fig 4 UPDATED — Efficiency Curve + Temporal Point
# ═══════════════════════════════════════════════════════════

# ── Load original data (same as M9) ──
eff_curve = pd.read_parquet(MODULES / 'm7_cascade/analysis/efficiency_curve_data.parquet')
cost_eff = pd.read_excel(MODULES / 'm7_cascade/analysis/cost_effectiveness.xlsx')

fig, ax = plt.subplots(figsize=(SINGLE_COL * 1.5, SINGLE_COL * 1.2))

# ── Efficiency curve (UNCHANGED from M9) ──
ec = eff_curve.copy()
ax.plot(ec['bio_pct'], ec['accuracy'], '-', color=C_BASE, lw=1.8, zorder=2)

strategies = cost_eff.copy()
marker_map = {
    'Pure CBC_Only (0% bio)':              ('o', C_GREY,   6, 'CBC Only (0% bio)'),
    'Sadece Excluded → BIO':               ('s', C_ORANGE, 7, 'Excluded only → BIO'),
    'MEDIUM + LOW → BIO':                  ('D', C_PURPLE, 7, 'MEDIUM + LOW → BIO'),
    'Excluded + MEDIUM + LOW → BIO':       ('*', C_HIGH,  12, 'Cascade (sweet spot)'),
    'Full BIO (100%)':                     ('^', C_GREEN,  7, 'Full BIO (100%)'),
}

for _, row in strategies.iterrows():
    name = row['strategy']
    if name in marker_map:
        m, c, ms, lbl = marker_map[name]
        ax.plot(row['bio_pct'], row['accuracy'], m, color=c,
                markersize=ms, zorder=5, markeredgecolor='#333333', markeredgewidth=0.5)

# Cascade annotation (UNCHANGED)
cascade_row = strategies[strategies['strategy'].str.contains('Excluded.*MEDIUM.*LOW', regex=True)]
if not cascade_row.empty:
    cr = cascade_row.iloc[0]
    ax.annotate(f'Cascade\n({cr["bio_pct"]:.0f}% bio, acc={cr["accuracy"]:.3f})',
                xy=(cr['bio_pct'], cr['accuracy']),
                xytext=(cr['bio_pct'] - 28, cr['accuracy'] + 0.032),
                fontsize=7, color=C_HIGH, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=C_HIGH, lw=0.8))

# Reference lines (UNCHANGED)
full_row = strategies[strategies['strategy'].str.contains('Full BIO')]
if not full_row.empty:
    fr = full_row.iloc[0]
    ax.axhline(y=fr['accuracy'], color=C_GREEN, ls=':', lw=0.7, alpha=0.6)
    ax.text(5, fr['accuracy'] + 0.005, f'Full BIO ({fr["accuracy"]:.3f})',
            fontsize=6.5, color=C_GREEN)

pure_row = strategies[strategies['strategy'].str.contains('Pure CBC')]
if not pure_row.empty:
    pr = pure_row.iloc[0]
    ax.axhline(y=pr['accuracy'], color=C_GREY, ls=':', lw=0.7, alpha=0.6)
    ax.text(5, pr['accuracy'] + 0.005, f'CBC Only ({pr["accuracy"]:.3f})',
            fontsize=6.5, color=C_GREY)

# Sweet spot shading (UNCHANGED)
if not cascade_row.empty:
    cr = cascade_row.iloc[0]
    ax.axvspan(cr['bio_pct'] - 8, cr['bio_pct'] + 8, alpha=0.08, color=C_HIGH, zorder=0)

# ═══ TEMPORAL OVERLAY (NEW) ═══
temp_bio_pct = cascade_metrics_dict['Bio Utilization (%)']
temp_acc = cascade_metrics_dict['Cascade Accuracy']
ax.plot(temp_bio_pct, temp_acc, TEMPORAL_MARKER, color=TEMPORAL_COLOR,
        markersize=10, markeredgecolor='#333333', markeredgewidth=0.8, zorder=7)

ax.annotate(f'Temporal\n({temp_bio_pct:.0f}% bio, acc={temp_acc:.3f})',
            xy=(temp_bio_pct, temp_acc),
            xytext=(temp_bio_pct - 30, temp_acc - 0.035),
            fontsize=7, color=TEMPORAL_COLOR, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=TEMPORAL_COLOR, lw=0.8))

# ── Axis & legend (UNCHANGED + temporal) ──
ax.set_xlabel('Biochemistry utilization (%)', fontsize=9)
ax.set_ylabel('E2E accuracy', fontsize=9)
ax.set_xlim(-3, 105)
ax.set_ylim(0.52, 0.82)  # extended slightly to fit temporal point
ax.tick_params(labelsize=7.5)

legend_items = [
    Line2D([0], [0], color=C_BASE, lw=1.5, label='Efficiency curve'),
    Line2D([0], [0], marker='o', color=C_GREY, lw=0, markersize=5, label='CBC Only (0% bio)'),
    Line2D([0], [0], marker='s', color=C_ORANGE, lw=0, markersize=5, label='Excluded only → BIO'),
    Line2D([0], [0], marker='D', color=C_PURPLE, lw=0, markersize=5, label='MEDIUM + LOW → BIO'),
    Line2D([0], [0], marker='*', color=C_HIGH, lw=0, markersize=9, label='Cascade (sweet spot)'),
    Line2D([0], [0], marker='^', color=C_GREEN, lw=0, markersize=6, label='Full BIO (100%)'),
    Line2D([0], [0], marker=TEMPORAL_MARKER, color=TEMPORAL_COLOR, lw=0,
           markersize=7, markeredgecolor='#333333', markeredgewidth=0.5, label='Temporal cascade'),
]
ax.legend(handles=legend_items, fontsize=6, loc='lower right')

fig.tight_layout()
save_fig(fig, FIG_DIR, 'fig4_efficiency_curve')
plt.show()
print("✅ Fig 4 updated with temporal cascade point")

In [ ]:
# ═══════════════════════════════════════════════════════════
# M11 · Cell 15a: Backup originals (optional)
# ═══════════════════════════════════════════════════════════
import shutil

BACKUP = PUB / 'pre_temporal_backup'
BACKUP.mkdir(exist_ok=True)

# Tables
if (TAB_DIR / 'ALL_PUBLICATION_TABLES.xlsx').exists():
    shutil.copy2(TAB_DIR / 'ALL_PUBLICATION_TABLES.xlsx',
                 BACKUP / 'ALL_PUBLICATION_TABLES_original.xlsx')

# Figures already overwritten — check if TIFF originals still exist
for sub in ['PNG_300DPI', 'TIFF_600DPI']:
    src = FIG_DIR / sub
    if src.exists():
        dst = BACKUP / sub
        dst.mkdir(parents=True, exist_ok=True)
        for f in src.iterdir():
            if f.name.startswith(('fig2_', 'fig3_', 'fig4_')):
                if not (dst / f.name).exists():
                    shutil.copy2(f, dst / f.name)

print(f"✅ Backup → {BACKUP}")
for f in sorted(BACKUP.rglob('*')):
    if f.is_file():
        print(f"  {f.relative_to(BACKUP)}")

#Tables

In [ ]:
# ═══════════════════════════════════════════════════════════
# Manual clean rebuild — ALL_PUBLICATION_TABLES from scratch
# ═══════════════════════════════════════════════════════════

# 1. Load M0–M9 source data (same as M9 notebook Cell 3)
cal_summary = pd.read_excel(MODULES / 'm1_calibration/metrics/calibration_summary.xlsx')
with open(MODULES / 'm2_thresholds/configs/threshold_config.json') as f:
    th_config_src = json.load(f)
op_points = pd.read_excel(MODULES / 'm2_thresholds/metrics/operating_points.xlsx')
zone_dist = pd.read_excel(MODULES / 'm2_thresholds/metrics/zone_distribution.xlsx')
per_class_test = pd.read_excel(MODULES / 'm3_errors/analysis/per_class_metrics.xlsx', sheet_name='Test')
misclass = pd.read_excel(MODULES / 'm3_errors/analysis/misclassification_patterns.xlsx')
e2e_comp = pd.read_excel(MODULES / 'm4_e2e/analysis/e2e_scenario_comparison.xlsx')
e2e_zone = pd.read_excel(MODULES / 'm4_e2e/analysis/e2e_zone_summary.xlsx')
conf_summary_src = pd.read_excel(MODULES / 'm6_conformal/metrics/conformal_summary.xlsx')
conf_class_cov = pd.read_excel(MODULES / 'm6_conformal/metrics/conformal_class_coverage.xlsx')
cascade_sim_test = pd.read_parquet(MODULES / 'm7_cascade/analysis/cascade_simulation.parquet')
cascade_metrics_test = pd.read_excel(MODULES / 'm7_cascade/analysis/cascade_metrics.xlsx')
cost_eff = pd.read_excel(MODULES / 'm7_cascade/analysis/cost_effectiveness.xlsx')
reflex_rec = pd.read_parquet(MODULES / 'm7_cascade/analysis/reflex_recommendations.parquet')

# 2. Build original (M9) tables
# Table 1 — Baseline (original 6 rows)
auc_map = {('CBC_Only',1):0.710, ('CBC_Only',2):0.883, ('CBC_BIO',1):0.869, ('CBC_BIO',2):0.930}
acc_map = {('CBC_Only',1):0.764, ('CBC_Only',2):0.667, ('CBC_BIO',1):0.838, ('CBC_BIO',2):0.770}

t1_rows = []
for sc in ['CBC_Only', 'CBC_BIO']:
    for st_name in ['Stage 1', 'Stage 2']:
        macro = per_class_test[(per_class_test['Scenario']==sc) & (per_class_test['Stage']==st_name) & (per_class_test['Class']=='MACRO AVG')]
        st_int = 1 if st_name=='Stage 1' else 2
        n_test = SCENARIOS[sc][f's{st_int}'].n_test
        if not macro.empty:
            r = macro.iloc[0]
            t1_rows.append({
                'Scenario': sc.replace('_',' '), 'Stage': st_name, 'N (Test)': n_test,
                'Accuracy': acc_map[(sc,st_int)],
                'Macro Precision': round(r['Precision'],3), 'Macro Recall': round(r['Recall'],3),
                'Macro F1': round(r['F1'],3), 'ROC AUC': auc_map[(sc,st_int)]
            })
for sc in ['CBC_Only', 'CBC_BIO']:
    er = e2e_comp[(e2e_comp['scenario']==sc) & (e2e_comp['split']=='test')]
    if not er.empty:
        r = er.iloc[0]
        t1_rows.append({
            'Scenario': sc.replace('_',' '), 'Stage': 'E2E', 'N (Test)': 229,
            'Accuracy': round(r['e2e_accuracy'],3),
            'Macro Precision': np.nan, 'Macro Recall': np.nan,
            'Macro F1': round(r['e2e_f1_macro'],3), 'ROC AUC': np.nan
        })

# Add temporal rows (N=110, real)
for sc in ['CBC_Only', 'CBC_BIO']:
    tr = temporal_baseline[(temporal_baseline['Scenario']==sc) & (temporal_baseline['Stage']=='Stage 2')].iloc[0]
    t1_rows.append({
        'Scenario': sc.replace('_',' '), 'Stage': 'Stage 2 (Temporal)', 'N (Test)': int(tr['N']),
        'Accuracy': tr['Accuracy'], 'Macro Precision': tr['Macro Precision'],
        'Macro Recall': tr['Macro Recall'], 'Macro F1': tr['Macro F1'], 'ROC AUC': tr['ROC AUC']
    })
te = temporal_baseline[temporal_baseline['Stage']=='E2E'].iloc[0]
t1_rows.append({
    'Scenario': 'Cascade', 'Stage': 'E2E (Temporal)', 'N (Test)': int(te['N']),
    'Accuracy': te['Accuracy'], 'Macro Precision': np.nan, 'Macro Recall': np.nan,
    'Macro F1': te['Macro F1'], 'ROC AUC': np.nan
})
t1 = pd.DataFrame(t1_rows)

# Table 2 — Calibration
t2 = cal_summary[['Scenario','Stage','Status','Brier','ECE','MCE','Selected']].copy()
t2.columns = ['Scenario','Stage','Status','Brier Score','ECE','MCE','Selected']
for c in ['Brier Score','ECE','MCE']:
    t2[c] = t2[c].round(4)
for _, r in temporal_calibration.iterrows():
    t2 = pd.concat([t2, pd.DataFrame([{
        'Scenario': r['Scenario'].replace('_',' '), 'Stage':'Stage 2','Status':'Temporal',
        'Brier Score':r['Brier Score'],'ECE':r['ECE'],'MCE':np.nan,'Selected':'-'
    }])], ignore_index=True)

# Table 3A, 3B — Op points & zones (unchanged)
t3a = op_points.copy()
for c in t3a.select_dtypes(include=[np.number]).columns:
    t3a[c] = t3a[c].round(3)
t3a = t3a.rename(columns={'scenario':'Scenario','s1_threshold':'S1 Threshold','s1_sensitivity':'S1 Sensitivity',
    's1_specificity':'S1 Specificity','s1_f1':'S1 F1','s1_accuracy':'S1 Accuracy',
    's2_zone_low':'S2 Zone LOW Boundary','s2_zone_high':'S2 Zone HIGH Boundary',
    's2_high_pct':'HIGH Zone (%)','s2_high_acc':'HIGH Zone Accuracy',
    'joint_accuracy':'Joint Accuracy','joint_coverage':'Joint Coverage'})
if 'Scenario' in t3a.columns:
    t3a['Scenario'] = t3a['Scenario'].astype(str).str.replace('_',' ')

t3b = zone_dist.copy()
for c in t3b.select_dtypes(include=[np.number]).columns:
    t3b[c] = t3b[c].round(3)
t3b = t3b.rename(columns={'scenario':'Scenario','class':'Class','zone':'Zone','n':'N',
    'pct_of_class':'Class Distribution (%)','accuracy':'Accuracy'})
if 'Scenario' in t3b.columns:
    t3b['Scenario'] = t3b['Scenario'].astype(str).str.replace('_',' ')

# Table 4A — Per-class (+ temporal)
t4 = per_class_test.copy()
for c in t4.select_dtypes(include=[np.number]).columns:
    t4[c] = t4[c].round(3)
temp_pc = temporal_perclass.copy()
temp_pc['Scenario'] = temp_pc['Scenario'].str.replace('_',' ')
t4 = pd.concat([t4, temp_pc], ignore_index=True)

# Table 4B — Misclassification
t4b = misclass.copy()
if 'Error_Count' in t4b.columns:
    t4b = t4b.sort_values('Error_Count', ascending=False)
for c in t4b.select_dtypes(include=[np.number]).columns:
    t4b[c] = t4b[c].round(3)
t4b = t4b.rename(columns={'True_Class':'True Class','Pred_Class':'Predicted Class',
    'Error_Count':'Error Count','Error_Pct':'Error Rate (%)',
    'Mean_Conf':'Mean Confidence','Median_Conf':'Median Confidence',
    'Min_Conf':'Min Confidence','Max_Conf':'Max Confidence'})

# Table 5A — E2E (+ temporal)
t5 = e2e_comp.copy()
for c in t5.select_dtypes(include=[np.number]).columns:
    t5[c] = t5[c].round(3)
t5 = t5.rename(columns={'scenario':'Scenario','split':'Split','e2e_accuracy':'E2E Accuracy',
    'e2e_f1_macro':'E2E Macro F1','lost_patients':'Lost Patients','extra_patients':'Extra Patients',
    'high_zone_pct':'HIGH Zone (%)','high_zone_acc':'HIGH Zone Accuracy'})
if 'Scenario' in t5.columns:
    t5['Scenario'] = t5['Scenario'].astype(str).str.replace('_',' ')
t5 = pd.concat([t5, pd.DataFrame([{
    'Scenario':'Cascade','Split':'Temporal',
    'E2E Accuracy':cascade_metrics_dict['Cascade Accuracy'],
    'E2E Macro F1':cascade_metrics_dict['Cascade Macro F1'],
    'Lost Patients':0,'Extra Patients':0,
    'HIGH Zone (%)':round(len(cascade[cascade['final_zone']=='HIGH'])/len(cascade)*100,1),
    'HIGH Zone Accuracy':round(cascade[cascade['final_zone']=='HIGH']['correct'].mean(),3)
}])], ignore_index=True)

# Table 6A — Conformal (+ temporal)
t6 = conf_summary_src[conf_summary_src['method']=='randomized'].copy()
t6 = t6[['scenario','stage','alpha','target_coverage','empirical_coverage','avg_set_size','singleton_rate']]
t6.columns = ['Scenario','Stage','α','Target Coverage','Empirical Coverage','Avg Set Size','Singleton Rate']
for c in t6.select_dtypes(include=[np.number]).columns:
    t6[c] = t6[c].round(3)
t6['Scenario'] = t6['Scenario'].str.replace('_',' ')
for _, r in temporal_conformal.iterrows():
    t6 = pd.concat([t6, pd.DataFrame([{
        'Scenario': r['Scenario'].replace('_',' ') + ' (Temporal)','Stage':'Stage 2','α':r['Alpha'],
        'Target Coverage':1-r['Alpha'],'Empirical Coverage':r['Coverage'],
        'Avg Set Size':r['Avg Set Size'],'Singleton Rate':r['Singleton Rate']
    }])], ignore_index=True)

t6b = conf_class_cov.copy()
t6b.columns = ['Scenario','α','Class','Coverage']
t6b['Class'] = t6b['Class'].replace({'DEA':'IDA','HGB_HTZ':'HGB HTZ'})  # display labels
t6b['Coverage'] = t6b['Coverage'].round(3)
t6b['Scenario'] = t6b['Scenario'].str.replace('_',' ')

# Table 7A — Cascade (+ temporal)
t7 = cascade_metrics_test.copy()
for k, v in cascade_metrics_dict.items():
    t7 = pd.concat([t7, pd.DataFrame([{'Metric': f'{k} (Temporal)','Value': v}])], ignore_index=True)

# Table 7B, 7C
t7b = cost_eff.copy()
for c in t7b.select_dtypes(include=[np.number]).columns:
    t7b[c] = t7b[c].round(3)
t7b = t7b.rename(columns={'strategy':'Strategy','accuracy':'E2E Accuracy',
    'bio_pct':'Biochemistry Utilization (%)','bio_count':'Patients Receiving Biochemistry'})

t7c_rows = []
for tier in sorted(cascade_sim_test['final_tier'].unique()):
    sub = cascade_sim_test[cascade_sim_test['final_tier']==tier]
    t7c_rows.append({'Final Tier': f"Tier {tier}",'N': len(sub),
        'Proportion (%)': round(len(sub)/len(cascade_sim_test)*100,1),
        'Accuracy': round(sub['correct'].mean(),3)})
t7c = pd.DataFrame(t7c_rows)

# Table 8A, 8B
# (reflex_test already in English in the parquet; no translation needed)
t8a = reflex_rec.groupby(['reflex_test','urgency']).agg(n=('patient_idx','count'), accuracy=('correct','mean')).reset_index()
t8a['pct'] = (t8a['n']/len(reflex_rec)*100).round(1)
t8a['accuracy'] = t8a['accuracy'].round(3)
t8a = t8a.sort_values('n', ascending=False)
t8a = t8a.rename(columns={'reflex_test':'Recommended Test','urgency':'Urgency',
    'n':'N','accuracy':'Accuracy','pct':'Proportion (%)'})

t8b = reflex_rec.groupby(['final_pred_label','reflex_test']).agg(n=('patient_idx','count'), accuracy=('correct','mean')).reset_index()
t8b['accuracy'] = t8b['accuracy'].round(3)
t8b = t8b.sort_values(['final_pred_label','n'], ascending=[True, False])
t8b = t8b.rename(columns={'final_pred_label':'Final Prediction','reflex_test':'Recommended Test',
    'n':'N','accuracy':'Accuracy'})
t8b['Final Prediction'] = t8b['Final Prediction'].replace(
    {'DEA':'IDA','DAS':'OAC','HGB_HTZ':'HGB HTZ','FP_NO_S2':'Unresolved'})  # display labels

# Table 9 (Temporal demographics)
demo_rows = []
for ci in range(4):
    cname = CLASS_NAMES_S2[ci]
    sub = temporal[temporal['target']==ci]
    demo_rows.append({
        'Class':{'DEA':'IDA','HGB_HTZ':'HGB HTZ'}.get(cname, cname),'N':len(sub),
        'Proportion (%)':round(len(sub)/len(temporal)*100,1),
        'Age (mean±SD)':f"{sub['yas'].mean():.1f} ± {sub['yas'].std():.1f}",
        'HGB (mean±SD)':f"{sub['hgb_g_d_l'].mean():.1f} ± {sub['hgb_g_d_l'].std():.1f}",
        'MCV (mean±SD)':f"{sub['mcv_f_l'].mean():.1f} ± {sub['mcv_f_l'].std():.1f}",
        'Ferritin (mean±SD)':f"{sub['ferritin'].mean():.1f} ± {sub['ferritin'].std():.1f}"
    })
demo_rows.append({
    'Class':'TOTAL','N':len(temporal),'Proportion (%)':100.0,
    'Age (mean±SD)':f"{temporal['yas'].mean():.1f} ± {temporal['yas'].std():.1f}",
    'HGB (mean±SD)':f"{temporal['hgb_g_d_l'].mean():.1f} ± {temporal['hgb_g_d_l'].std():.1f}",
    'MCV (mean±SD)':f"{temporal['mcv_f_l'].mean():.1f} ± {temporal['mcv_f_l'].std():.1f}",
    'Ferritin (mean±SD)':f"{temporal['ferritin'].mean():.1f} ± {temporal['ferritin'].std():.1f}"
})
t9 = pd.DataFrame(demo_rows)

# Thesis comparison (from M9 — unchanged)
thesis_comp_path = TAB_DIR / 'Thesis_vs_M9_comparison.xlsx'
thesis_comp = pd.read_excel(thesis_comp_path) if thesis_comp_path.exists() else None

# ═══ Save all individual files + combined ═══
# Individual files
t1.to_excel(TAB_DIR / 'Table_1_baseline_performance.xlsx', index=False)
t2.to_excel(TAB_DIR / 'Table_2_calibration_metrics.xlsx', index=False)
with pd.ExcelWriter(TAB_DIR / 'Table_3_operating_points_zones.xlsx') as w:
    t3a.to_excel(w, sheet_name='Operating_Points', index=False)
    t3b.to_excel(w, sheet_name='Zone_Distribution', index=False)
with pd.ExcelWriter(TAB_DIR / 'Table_4_error_analysis.xlsx') as w:
    t4.to_excel(w, sheet_name='Per_Class_Metrics', index=False)
    t4b.to_excel(w, sheet_name='Misclassification_Patterns', index=False)
t5.to_excel(TAB_DIR / 'Table_5_e2e_performance.xlsx', index=False)
with pd.ExcelWriter(TAB_DIR / 'Table_6_conformal_prediction.xlsx') as w:
    t6.to_excel(w, sheet_name='Summary', index=False)
    t6b.to_excel(w, sheet_name='Class_Coverage', index=False)
with pd.ExcelWriter(TAB_DIR / 'Table_7_cascade_simulation.xlsx') as w:
    t7.to_excel(w, sheet_name='Cascade_Metrics', index=False)
    t7b.to_excel(w, sheet_name='Cost_Effectiveness', index=False)
    t7c.to_excel(w, sheet_name='Tier_Summary', index=False)
with pd.ExcelWriter(TAB_DIR / 'Table_8_reflex_tests.xlsx') as w:
    t8a.to_excel(w, sheet_name='Distribution', index=False)
    t8b.to_excel(w, sheet_name='By_Prediction', index=False)
t9.to_excel(TAB_DIR / 'Table_9_temporal_cohort.xlsx', index=False)
comparison.to_excel(TAB_DIR / 'Temporal_vs_Test_comparison.xlsx', index=False)
shift_df.to_excel(TAB_DIR / 'Table_S1_feature_shift.xlsx', index=False)

# Combined workbook
with pd.ExcelWriter(TAB_DIR / 'ALL_PUBLICATION_TABLES.xlsx', engine='openpyxl') as w:
    t1.to_excel(w, sheet_name='Table_1_Baseline', index=False)
    t2.to_excel(w, sheet_name='Table_2_Calibration', index=False)
    t3a.to_excel(w, sheet_name='Table_3A_Op_Points', index=False)
    t3b.to_excel(w, sheet_name='Table_3B_Zones', index=False)
    t4.to_excel(w, sheet_name='Table_4A_PerClass', index=False)
    t4b.to_excel(w, sheet_name='Table_4B_Misclass', index=False)
    t5.to_excel(w, sheet_name='Table_5A_E2E_Comp', index=False)
    t6.to_excel(w, sheet_name='Table_6A_Conformal', index=False)
    t6b.to_excel(w, sheet_name='Table_6B_ClassCov', index=False)
    t7.to_excel(w, sheet_name='Table_7A_Cascade', index=False)
    t7b.to_excel(w, sheet_name='Table_7B_CostEff', index=False)
    t7c.to_excel(w, sheet_name='Table_7C_TierFlow', index=False)
    t8a.to_excel(w, sheet_name='Table_8A_Reflex', index=False)
    t8b.to_excel(w, sheet_name='Table_8B_ReflexPred', index=False)
    t9.to_excel(w, sheet_name='Table_9_Temporal', index=False)
    comparison.to_excel(w, sheet_name='Test_vs_Temporal', index=False)
    if thesis_comp is not None:
        thesis_comp.to_excel(w, sheet_name='Thesis_vs_M9', index=False)

# Verify
print("✅ All tables rebuilt from scratch with REAL temporal data (N=110)\n")
print("Table 1 — Final:")
print(t1.to_string(index=False))
print(f"\n✅ ALL_PUBLICATION_TABLES.xlsx — {len(pd.ExcelFile(TAB_DIR / 'ALL_PUBLICATION_TABLES.xlsx').sheet_names)} sheets")

## Final VErification

In [ ]:
# ═══════════════════════════════════════════════════════════
# M11 · Cell 16: FINAL VERIFICATION
# ═══════════════════════════════════════════════════════════
import os

print("=" * 70)
print("M11 TEMPORAL VALIDATION — FINAL INVENTORY")
print("=" * 70)

# ── 1. M11 Module Files ──
print("\n1. M11 MODULE FILES")
print("-" * 50)
for root, dirs, files in os.walk(M11_DIR):
    for f in sorted(files):
        fp = os.path.join(root, f)
        size = os.path.getsize(fp) / 1024
        rel = os.path.relpath(fp, M11_DIR)
        print(f"  {'✅'} {rel} ({size:.1f} KB)")

# ── 2. Updated Publication Tables ──
print("\n2. PUBLICATION TABLES")
print("-" * 50)
for f in sorted(TAB_DIR.iterdir()):
    if f.is_file():
        size = f.stat().st_size / 1024
        print(f"  📋 {f.name} ({size:.1f} KB)")

# ── 3. Updated Publication Figures ──
print("\n3. PUBLICATION FIGURES")
print("-" * 50)
for sub in ['PNG_300DPI', 'TIFF_600DPI']:
    subdir = FIG_DIR / sub
    if subdir.exists():
        print(f"  [{sub}]")
        for f in sorted(subdir.iterdir()):
            size = f.stat().st_size / 1024
            # Mark updated vs new
            tag = "🆕" if 'temporal' in f.name or 's6' in f.name or 's7' in f.name else "🔄"
            if f.name.startswith(('fig2_', 'fig3_', 'fig4_')):
                tag = "🔄"
            print(f"    {tag} {f.name} ({size:.1f} KB)")

# ── 4. Backup Check ──
print("\n4. PRE-TEMPORAL BACKUP")
print("-" * 50)
backup = PUB / 'pre_temporal_backup'
if backup.exists():
    for f in sorted(backup.rglob('*')):
        if f.is_file():
            print(f"  💾 {f.relative_to(backup)}")
else:
    print("  ⚠️ No backup found")

# ── 5. Key Metrics Summary ──
print("\n5. KEY METRICS SUMMARY (Temporal Cohort)")
print("-" * 50)
print(f"  Cohort size:           {len(cascade)}")
_cls_disp = ['IDA', 'HA', 'HGB HTZ', 'Normal']  # display labels
_cls_counts = cascade['target'].value_counts().sort_index().tolist()
print(f"  Class distribution:    {dict(zip(_cls_disp, _cls_counts))}")
print(f"  S2 CBC Only accuracy:  {temporal_baseline[temporal_baseline['Scenario']=='CBC_Only']['Accuracy'].values[0]:.3f}")
print(f"  S2 CBC+BIO accuracy:   {temporal_baseline[temporal_baseline['Scenario']=='CBC_BIO']['Accuracy'].values[0]:.3f}")
print(f"  Cascade E2E accuracy:  {cascade_metrics_dict['Cascade Accuracy']}")
print(f"  Cascade Macro F1:      {cascade_metrics_dict['Cascade Macro F1']}")
print(f"  Tier 1 retained:       {cascade_metrics_dict['Tier 1 Retained (%)']:.1f}%")
print(f"  Bio utilization:       {cascade_metrics_dict['Bio Utilization (%)']:.1f}%")
print(f"  Conformal CBC+BIO cov: {temporal_conformal[(temporal_conformal['Scenario']=='CBC_BIO') & (temporal_conformal['Alpha']==0.10)]['Coverage'].values[0]:.3f}")
print(f"  Feature shifts:        {(shift_df['Verdict'].str.contains('Significant')).sum()} significant, "
      f"{(shift_df['Verdict'].str.contains('Minor')).sum()} minor, "
      f"{(shift_df['Verdict']=='✅ Stable').sum()} stable")

# ── 6. Readiness Check ──
print(f"\n{'='*70}")
print("READINESS FOR REAL TEMPORAL DATA")
print("=" * 70)
print("""
  When real temporal data arrives:
  1. Replace temporal_cohort.parquet with real data (same columns)
  2. Re-run Cells 4 → 5 → 5b → 6 → 7 → 8 → 10–15
  3. All tables & figures auto-update

  Required columns in real data:
    - target (int): 0=IDA, 1=HA, 2=HGB HTZ, 3=Normal (IDA stored internally as 'DEA')
    - target_label (str): class name
    - diagnosis (str): clinical diagnosis
    - All 17 base features + ratios auto-computed
    - patient_idx, cohort columns auto-added
""")
print("✅ M11 TEMPORAL VALIDATION PIPELINE COMPLETE")

##

##